# 3. Final Pipeline: Run Everything and Build the Deliverable

This notebook is the single entry point for the project. It

1. **runs every notebook** end-to-end, in order — feature engineering (`1_feature_engineering`) followed by the four per-instrument meta-models (`2a`–`2d`);
2. **assembles the one required CSV deliverable**, `data/deliverables/predictions.csv`, in the exact `date,instrument,prediction` format from the coursework brief, covering H1 2022 for the full **Energy** asset class (`cl1s`, `ho1s`, `rb1s`, `ng1s`);
3. **shows the summary tables and graphs** — the winning model per instrument with its walk-forward CV AUC and out-of-sample AUC, prediction coverage, the prediction time series, and the prediction distributions.

Each per-instrument notebook writes its own `predictions_<ticker>.csv` as a side effect of execution; this notebook merges those into the combined deliverable. The energy asset class satisfies the brief's requirement to cover at least one full asset class.

## 1. Run the full pipeline

In [1]:
import sys, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# When run with Jupyter, the working directory is the notebooks/ folder.
NB_DIR   = Path.cwd()
REPO     = NB_DIR.parent
DELIV    = REPO / 'data' / 'deliverables'
EXEC_DIR = NB_DIR / '_executed'          # executed copies land here; source notebooks stay clean

# Run order: feature engineering first (produces features.parquet), then the four meta-models.
PIPELINE = [
    '1_feature_engineering.ipynb',
    '2a_rbob_gasoline_model_training.ipynb',
    '2b_heating_oil_model_training.ipynb',
    '2c_natural_gas_model_training.ipynb',
    '2d_wti_crude_oil_model_training.ipynb',
]

# Energy asset class = the instruments we cover end-to-end.
INSTRUMENTS = {
    'cl1s': 'WTI Crude Oil',
    'ho1s': 'Heating Oil',
    'rb1s': 'RBOB Gasoline',
    'ng1s': 'Natural Gas',
}

# Set False to skip the (slow) re-execution and just rebuild the deliverable + plots from the
# prediction CSVs already on disk. Set True for a clean end-to-end reproducibility run.
RUN_NOTEBOOKS  = True
CELL_TIMEOUT_S = 3600   # per-cell timeout; the neural-net training notebooks are the slow ones

print('repo        :', REPO)
print('deliverables:', DELIV)
print('run order   :', PIPELINE)

repo        : /Users/adba/Documents/icl/modules/systematic_trading_strategies/5_coursework/systematic_finance_2026_project
deliverables: /Users/adba/Documents/icl/modules/systematic_trading_strategies/5_coursework/systematic_finance_2026_project/data/deliverables
run order   : ['1_feature_engineering.ipynb', '2a_rbob_gasoline_model_training.ipynb', '2b_heating_oil_model_training.ipynb', '2c_natural_gas_model_training.ipynb', '2d_wti_crude_oil_model_training.ipynb']


We execute each notebook with `jupyter nbconvert --execute`. Every notebook runs with its own directory as the working directory, so the `../data/...` relative paths resolve correctly. The executed copies are written to `notebooks/_executed/` to keep the source notebooks clean; the real outputs we care about (`features.parquet` and the `predictions_<ticker>.csv` files) are produced as side effects regardless of where the rendered notebook is saved.

If a notebook fails (for example a missing dependency), we record it and carry on, then fall back to any prediction CSVs already on disk so the deliverable can still be assembled.

In [2]:
status = []
if RUN_NOTEBOOKS:
    EXEC_DIR.mkdir(exist_ok=True)
    for nb in PIPELINE:
        t0 = time.time()
        cmd = [
            sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
            str(NB_DIR / nb), '--output-dir', str(EXEC_DIR),
            f'--ExecutePreprocessor.timeout={CELL_TIMEOUT_S}',
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        ok = proc.returncode == 0
        mins = round((time.time() - t0) / 60, 1)
        status.append({'notebook': nb, 'status': 'ok' if ok else 'FAILED', 'minutes': mins})
        print(f"{'ok  ' if ok else 'FAIL'}  {nb:<42}  {mins:>5} min")
        if not ok:
            print('  --- last lines of stderr ---')
            print('\n'.join(proc.stderr.strip().splitlines()[-15:]))
else:
    print('RUN_NOTEBOOKS = False  ->  skipping execution, using existing CSVs in', DELIV)

run_log = pd.DataFrame(status)
run_log

FAIL  1_feature_engineering.ipynb                   0.1 min
  --- last lines of stderr ---
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
------------------

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
/var/folders/m8/z4210p954l9gspqv294cnshc0000gp/T/ipykernel_97594/2847931983.py in <module>
     13 from sklearn.mixture import GaussianMixture
     14 from sklearn.preprocessing import StandardScaler
---> 15 from hmmlearn.hmm import GaussianHMM

ModuleNotFoundError: No module named 'hmmlearn'
ModuleNotFoundError: No module named 'hmmlearn'
FAIL  2a_rbob_gasoline_model_training.ipynb         0.1 min
  --- last lines of stderr ---
np.random.seed(42)
tf.random.set_seed(42)
------------------

---------------------------------------------------------------------------
M

KeyboardInterrupt: 

## 2. Assemble the combined deliverable CSV

We read each per-instrument `predictions_<ticker>.csv`, concatenate them, validate the format required by the brief (`date,instrument,prediction`, one row per pair, probability in `[0, 1]`, covering H1 2022), sort by `(date, instrument)`, and write the single combined `predictions.csv`.

In [ ]:
frames = []
for ins in INSTRUMENTS:
    f = DELIV / f'predictions_{ins}.csv'
    if not f.exists():
        print('WARNING: missing', f.name, '- skipping')
        continue
    frames.append(pd.read_csv(f, parse_dates=['date']))

combined = pd.concat(frames, ignore_index=True)

# --- validate the deliverable format ---
assert list(combined.columns) == ['date', 'instrument', 'prediction'], combined.columns.tolist()
assert combined['prediction'].between(0, 1).all(), 'predictions must lie in [0, 1]'
assert not combined.duplicated(['date', 'instrument']).any(), 'one row per (date, instrument)'
assert (combined['date'].dt.year == 2022).all() and (combined['date'].dt.month <= 6).all(), 'H1 2022 only'

combined = combined.sort_values(['date', 'instrument']).reset_index(drop=True)
combined['date'] = combined['date'].dt.strftime('%Y-%m-%d')

out_path = DELIV / 'predictions.csv'
combined.to_csv(out_path, index=False)
print(f'Wrote {len(combined)} rows to {out_path}')
print('Instruments :', sorted(combined["instrument"].unique()))
print('Date range  :', combined["date"].min(), '->', combined["date"].max())
combined.head(8)

## 3. Winning model per instrument

The table below records, for each instrument, the model selected in that notebook's Section 11 along with its **walk-forward CV AUC** (mean across the time folds, the primary selection metric) and its **out-of-sample test AUC** on the H1 2022 holdout. The figures are taken from each notebook's model-selection write-up. Gasoline and heating oil are won by the **LSTM**; natural gas and WTI by **Logistic Regression** (selected under our complexity-penalising tie-break).

In [ ]:
winners = pd.DataFrame([
    {'instrument': 'rb1s', 'name': 'RBOB Gasoline', 'winning_model': 'LSTM',                'cv_auc': 0.7555, 'oos_auc': 0.5528},
    {'instrument': 'ho1s', 'name': 'Heating Oil',   'winning_model': 'LSTM',                'cv_auc': 0.7500, 'oos_auc': 0.5494},
    {'instrument': 'ng1s', 'name': 'Natural Gas',   'winning_model': 'Logistic Regression', 'cv_auc': 0.8889, 'oos_auc': 0.5752},
    {'instrument': 'cl1s', 'name': 'WTI Crude Oil', 'winning_model': 'Logistic Regression', 'cv_auc': 0.6654, 'oos_auc': 0.7548},
])
print(winners.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(winners))
ax.barh(y + 0.2, winners['cv_auc'],  height=0.4, color='steelblue', alpha=0.85, label='Walk-forward CV AUC')
ax.barh(y - 0.2, winners['oos_auc'], height=0.4, color='indianred', alpha=0.85, label='Out-of-sample AUC')
ax.axvline(0.5, color='black', lw=0.8, linestyle=':', label='random (0.50)')
ax.set_yticks(y); ax.set_yticklabels(winners['name'] + '\n(' + winners['winning_model'] + ')')
ax.set_xlabel('AUC'); ax.set_xlim(0, 1)
ax.set_title('Winning model per instrument: CV vs out-of-sample AUC')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 4. Prediction tables and graphs

### 4.1 Coverage and distribution per instrument

A quick check that every instrument covers the same H1 2022 calendar, plus summary statistics of the exported probabilities.

In [ ]:
dl = combined.copy()
dl['date'] = pd.to_datetime(dl['date'])
coverage = (dl.groupby('instrument')['prediction']
              .agg(rows='size', first_date=lambda s: dl.loc[s.index, 'date'].min().date(),
                   last_date=lambda s: dl.loc[s.index, 'date'].max().date(),
                   mean='mean', min='min', max='max')
              .round(4))
coverage

### 4.2 Meta-model probability through H1 2022

In [ ]:
piv = dl.pivot(index='date', columns='instrument', values='prediction').sort_index()

fig, axes = plt.subplots(len(INSTRUMENTS), 1, figsize=(12, 9), sharex=True)
for ax, ins in zip(axes, INSTRUMENTS):
    ax.plot(piv.index, piv[ins], color='steelblue', lw=1.3)
    ax.axhline(0.5, color='black', lw=0.8, linestyle=':', alpha=0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel(ins)
    ax.set_title(f'{INSTRUMENTS[ins]} ({ins}) — P(signal worth taking)', fontsize=10, loc='left')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('date')
plt.suptitle('Meta-model probabilities, H1 2022', y=1.005)
plt.tight_layout(); plt.show()

### 4.3 Distribution of exported probabilities

In [ ]:
fig, axes = plt.subplots(1, len(INSTRUMENTS), figsize=(16, 4), sharey=True)
for ax, ins in zip(axes, INSTRUMENTS):
    ax.hist(piv[ins].dropna(), bins=20, range=(0, 1), color='steelblue', alpha=0.85)
    ax.axvline(piv[ins].mean(), color='indianred', lw=1.5, label=f'mean {piv[ins].mean():.2f}')
    ax.axvline(0.5, color='black', lw=0.8, linestyle=':')
    ax.set_title(f'{ins}'); ax.set_xlabel('prediction'); ax.legend(fontsize=8)
axes[0].set_ylabel('count')
plt.suptitle('Distribution of meta-model probabilities by instrument', y=1.02)
plt.tight_layout(); plt.show()

## Summary

- **`data/deliverables/predictions.csv`** is the single required deliverable: one row per `(date, instrument, prediction)` for the Energy asset class across H1 2022, ready for the hidden H2 2022 rerun.
- The pipeline is reproducible end-to-end: set `RUN_NOTEBOOKS = True` and run this notebook top to bottom to regenerate features, retrain every meta-model, and rebuild the deliverable.
- Per the brief, selection is judged on methodology, not performance: gasoline and heating oil favour the sequence-aware **LSTM**, while natural gas and WTI favour the simpler **Logistic Regression** once we penalise complexity. Out-of-sample AUCs above 0.50 (notably WTI at 0.75) indicate the meta-model carries usable signal on the holdout, while values near 0.50 are reported honestly as marginal.